In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
import scipy
import sklearn
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
# root dir
root = 'SRM-RS/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,age,sex,ravlt_1,ravlt_5,ravlt_tot,ravlt_imm,ravlt_del,ravlt_rec,ravlt_fp,...,tmt_2,tmt_3,tmt_4,cw_1,cw_2,cw_3,cw_4,vf_1,vf_2,vf_3
0,sub-001,29,f,8,14,64,13,15,15,1,...,20,23.0,57.0,33,24,82,79,45.0,44.0,15.0
1,sub-002,29,f,8,14,65,13,14,15,1,...,23,33.0,101.0,34,29,44,51,32.0,36.0,14.0
2,sub-003,62,f,6,13,48,11,9,12,0,...,45,43.0,75.0,27,28,93,61,NaN,NaN,NaN
3,sub-004,20,f,8,14,62,13,13,15,0,...,18,17.0,49.0,28,20,43,45,56.0,54.0,23.0
4,sub-005,32,f,13,15,73,15,15,15,0,...,26,19.0,47.0,25,21,42,41,50.0,59.0,19.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,sub-107,18,f,8,15,65,15,15,15,0,...,19,24.0,39.0,21,19,41,44,42.0,54.0,12.0
107,sub-108,50,f,7,14,54,10,12,14,0,...,36,38.0,86.0,31,21,55,60,35.0,44.0,15.0
108,sub-109,19,f,8,15,61,14,15,15,0,...,20,38.0,128.0,40,20,90,84,30.0,38.0,13.0
109,sub-110,39,f,9,15,66,15,14,15,0,...,20,18.0,46.0,30,17,41,60,63.0,48.0,15.0


## Some data test, we found sub-29 cannot be read

In [5]:
"""# Test for bad channels, sampling freq and shape
derivative_path = root + 'derivatives/cleaned_epochs'
bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(derivative_path):
    if 'sub-' in sub:
        sub_path = os.path.join(derivative_path, sub)
        for session in os.listdir(sub_path): 
            session_path = os.path.join(sub_path, session, 'eeg/')
            # print(sub_path)
            for file in os.listdir(session_path):
                if '.set' in file:
                    file_path = os.path.join(session_path, file)
                    try:
                        raw = mne.io.read_epochs_eeglab(file_path)
                        # print(raw.get_montage())
                        # get bad channels
                        bad_channel = raw.info['bads']
                        bad_channel_list.append(bad_channel)
                        # get sampling frequency
                        sampling_freq = raw.info['sfreq']
                        sampling_freq_list.append(sampling_freq)
                        # get eeg data
                        data = raw.get_data()
                        data_shape = data.shape
                        data_shape_list.append(data_shape)
                    except Exception as e:
                        print(f"Failed to load {file_path}: {e}")
                        continue 
# Subject 29 and 104 cannot be loaded"""

Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-001\ses-t1\eeg\sub-001_ses-t1_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
53 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-002\ses-t1\eeg\sub-002_ses-t1_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
55 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-002\ses-t2\eeg\sub-002_ses-t2_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
55 matching events found
No baseline correction applied
0 projection items activated
Ready.
Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-3\SRM-RS\SRM-RS\

In [6]:
"""from collections import Counter

print(bad_channel_list)
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]
Channel number counter: Counter({55: 143, 53: 5, 22: 1, 48: 1, 43: 1})
Sampling rate counter: Counter({1024.0: 151})


## Features

In [5]:
def data_preprocessing(
    raw: mne.io.Raw,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")             
        
    return raw

In [6]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [8]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")

derivative_path = root + 'derivatives/cleaned_epochs'
sub_id = 1
for sub in os.listdir(derivative_path):
    if 'sub-' in sub:  
        if sub == "sub-029" or sub == "sub-104":  # skip subject 29 and subject 104
            continue
        feature_list = []
        sub_path = os.path.join(derivative_path, sub)
        for session in os.listdir(sub_path): 
            session_path = os.path.join(sub_path, session, 'eeg/')
            # print(session_path)
            for file in os.listdir(session_path):
                if '.set' in file:
                    file_path = os.path.join(session_path, file)
                    raw = mne.io.read_epochs_eeglab(file_path)
                    freq = raw.info['sfreq']
                    # 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
                    # For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
                    # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                    standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                    raw.pick(standard_channels)
                    info = raw.info
                    # get eeg data
                    data = np.transpose(raw.get_data(), (0, 2, 1))
                    print("Raw data shape ", data.shape)
                    # concatenate data to 2D array
                    data_concat = data.reshape(-1, data.shape[2])
                    raw = mne.io.RawArray(data_concat.T, info)
                    T_raw = raw.n_times
                    original_fs = raw.info['sfreq']
                    for fs in SAMPLE_RATE_LIST:
                        T_res = int(T_raw * fs / original_fs)
                        # compute number of segments
                        n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                        subject_segment_counts[sub_id][fs] += n_seg
                        N_total += n_seg
                        print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                        print("----------------------------------------")
        sub_id += 1
    print("-------------------------------------\n")
    
print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")
# Subject 29 cannot be loaded

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-001\ses-t1\eeg\sub-001_ses-t1_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
53 matching events found
No baseline correction applied
0 projection items activated
Ready.
Raw data shape  (53, 4096, 19)
Creating RawArray with float64 data, n_channels=19, n_times=217088
    Range : 0 ... 217087 =      0.000 ...   211.999 secs
Ready.
fs=200Hz: T_res=42400, STEP=200, n_seg=211
----------------------------------------
fs=100Hz: T_res=21200, STEP=200, n_seg=105
----------------------------------------
fs=50Hz: T_res=10600, STEP=200, n_seg=52
----------------------------------------
-------------------------------------

Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-002\s

## PASS 2: Process and store segments into memmap

In [13]:
output_root = os.path.join('Processed', sub_folder_path, 'SRM-RS')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\SRM-RS\X.dat
y path: Processed\L400\SRM-RS\y.dat


## PASS 2: Process and store segments into memmap

In [14]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation

derivative_path = root + 'derivatives/cleaned_epochs'
sub_id = 1
for sub in os.listdir(derivative_path):
    if 'sub-' in sub:  
        if sub == "sub-029" or sub == "sub-104":  # skip subject 29 and subject 104
            continue
        sub_label = 0
        sub_path = os.path.join(derivative_path, sub)
        for session in os.listdir(sub_path): 
            session_path = os.path.join(sub_path, session, 'eeg/')
            # print(session_path)
            for file in os.listdir(session_path):
                if '.set' in file:
                    file_path = os.path.join(session_path, file)
                    raw = mne.io.read_epochs_eeglab(file_path)
                    freq = raw.info['sfreq']
                    # 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
                    # For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
                    # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                    standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                         'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                    raw.pick(standard_channels)
                    info = raw.info
                    # get eeg data
                    data = np.transpose(raw.get_data(), (0, 2, 1))
                    print("Raw data shape ", data.shape)
                    # concatenate data to 2D array
                    data_concat = data.reshape(-1, data.shape[2])
                    raw = mne.io.RawArray(data_concat.T, info)
                    T_raw = raw.n_times
                    original_fs = raw.info['sfreq']
                    total_seconds_all += T_raw / original_fs
                    data = raw.get_data().T.astype('float32')  # (C, T) -> (T, C)
                    for fs in SAMPLE_RATE_LIST:
                        # resample to target fs
                        data_resampled = resample_time_series(data, original_fs, fs)
                        T_res, _ = data_resampled.shape
                        print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
            
                        # overlapped sliding window with fixed STEP (in timestamps)
                        starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                        print(f"fs={fs}Hz: segments={len(starts)}")
            
                        for s in starts:
                            if cur >= N_total:
                                raise RuntimeError("Exceeded predicted N_total.")
            
                            X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                            y_mm[cur, 0] = float(sub_label)       # label
                            y_mm[cur, 1] = float(sub_id)      # Global trial ID
                            y_mm[cur, 2] = float(fs)          # sample rate (scale)
                            cur += 1
        sub_id += 1
    print("-------------------------------------\n")
    
total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)
# Subject 29 cannot be loaded

-------------------------------------

-------------------------------------

Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-001\ses-t1\eeg\sub-001_ses-t1_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
53 matching events found
No baseline correction applied
0 projection items activated
Ready.
Raw data shape  (53, 4096, 19)
Creating RawArray with float64 data, n_channels=19, n_times=217088
    Range : 0 ... 217087 =      0.000 ...   211.999 secs
Ready.
fs=200Hz: resampled shape=(42400, 19)
fs=200Hz: segments=211
fs=100Hz: resampled shape=(21200, 19)
fs=100Hz: segments=105
fs=50Hz: resampled shape=(10600, 19)
fs=50Hz: segments=52
-------------------------------------

Extracting parameters from D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\SRM-RS\SRM-RS\derivatives\cleaned_epochs\sub-002\ses-t1\eeg\sub-002_ses-t1_task-resteyesc_desc-epochs_eeg.set...
Not setting metadata
55 matching events found
No

## Load and check the processed data

In [15]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 56880
  T = 400
  C = 19
  X_path = Processed\L400\SRM-RS\X.dat
  y_path = Processed\L400\SRM-RS\y.dat
-------------------------------------
Subject ID 001: X shape=(368, 400, 19), y shape=(368, 3)
Subject ID 002: X shape=(764, 400, 19), y shape=(764, 3)
Subject ID 003: X shape=(533, 400, 19), y shape=(533, 3)
Subject ID 004: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 005: X shape=(764, 400, 19), y shape=(764, 3)
Subject ID 006: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 007: X shape=(750, 400, 19), y shape=(750, 3)
Subject ID 008: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 009: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 010: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 011: X shape=(764, 400, 19), y shape=(764, 3)
Subject ID 012: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 013: X shape=(368, 400, 19), y shape=(368, 3)
Subject ID 014: X shape=(382, 400, 19), y shape=(382, 3)
Subject ID 015: X shape=(382, 400, 19), y shape=(